# Vilex — Kaggle Full Pipeline (Vertex AI + OmniVoice)

**Repo:** https://github.com/thaiphu05/Vilex  
**Model:** `gemini-3.6-flash` (Vertex AI via `GEMINI_CREDENTIALS`)  
**Pipeline:** Stage 1 speechify → Stage 2/4 synthesis → Stage 4 add_bc → Stage 5 OmniVoice TTS

> ⚠️ **2-phase execution required** — Kaggle has one interpreter. Stages 1-4 need `transformers>=4.53`, Stage 5 (OmniVoice) conflicts with vendored Chatterbox pin `4.46.3` (`requirements-stage5.txt`). Run Phase 1, then **Kernel → Restart & Clear Output**, then Phase 2.

## Kaggle Settings (do once)
- `Settings → Internet ON` (required for Gemini + git clone)
- `Settings → Accelerator → GPU T4 x2` (Stage 5 needs GPU, Stage 1-4 can run CPU but GPU faster)
- `Add-ons → Secrets` → add secret `GEMINI_CREDENTIALS_JSON` = paste entire service-account JSON (single line or pretty). Optional: `GEMINI_LOCATION` (default `global`, see `src/gemini_client.py:198`).
- Upload `voice_clone/` as Kaggle Dataset (each `*.wav` must have sidecar `*.txt` — see `tts_render/convert_spoken.py:715`). Mount it, note its path (e.g. `/kaggle/input/voice-clone`).
- `Save Version` after Phase 1 if you want to persist `outputs/` across restarts (or keep in `/kaggle/working` if you restart without clearing disk).


---
## PHASE 1 — Stages 1-4 (Dialogue Generation)
Run these cells top-to-bottom, then **Restart kernel** before Phase 2.


In [ ]:
# Cell 2 — Clone repo
!git clone https://github.com/thaiphu05/Vilex.git
%cd Vilex
!pwd && git log --oneline -3
!ls -lh

In [ ]:
# Cell 3 — Install Stages 1-4 deps
# Kaggle image already has torch; we install pipeline + Vertex helper
!pip install -q -r requirements.txt
!pip install -q google-genai google-auth
!python -c "import transformers, torch; print(f'transformers={transformers.__version__} torch={torch.__version__} cuda={torch.cuda.is_available()}')"

In [ ]:
# Cell 4 — Vertex AI auth (service-account JSON from Kaggle Secrets)
# GEMINI_CREDENTIALS_JSON = full JSON content of SA file (paste in Kaggle Secrets).
# GEMINI_LOCATION optional (global/us-central1/...). src/gemini_client.py uses GEMINI_CREDENTIALS or GOOGLE_APPLICATION_CREDENTIALS.
import os, json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
sa_json = secrets.get_secret("GEMINI_CREDENTIALS_JSON")
if not sa_json or not sa_json.strip():
    raise SystemExit("Missing Kaggle Secret GEMINI_CREDENTIALS_JSON — Add-ons → Secrets → paste SA JSON")

# Handle both raw JSON and base64-encoded JSON
import base64
sa_text = sa_json.strip()
if not sa_text.lstrip().startswith("{"):
    try:
        sa_text = base64.b64decode(sa_text).decode("utf-8")
    except Exception:
        pass

sa_path = Path("sa.json").resolve()
sa_path.write_text(sa_text, encoding="utf-8")
os.environ["GEMINI_CREDENTIALS"] = str(sa_path)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(sa_path)

try:
    loc = secrets.get_secret("GEMINI_LOCATION")
except Exception:
    loc = None
os.environ["GEMINI_LOCATION"] = (loc.strip() if loc and loc.strip() else "global")

info = json.loads(sa_text)
print(f"SA project: {info.get('project_id')}  location: {os.environ['GEMINI_LOCATION']}")
print(f"Credentials → {sa_path} ({sa_path.stat().st_size} bytes)")
!ls -lh sa.json

In [ ]:
# Cell 5 — Config knobs (edit as needed)
MODEL = "gemini-3.6-flash"
DATASET = "interviewer"
SPLIT = "train"
MAX_TRAIN_SAMPLES = 1   # Stage 1: số dialogue nguồn (1 = test nhanh, trống = all)
MAX_DIALOGUES = 1       # Stage 4: số dialogue synthesis
MAX_TURNS = 10
print(f"MODEL={MODEL} DATASET={DATASET} SPLIT={SPLIT} MAX_TRAIN={MAX_TRAIN_SAMPLES} MAX_DLG={MAX_DIALOGUES} MAX_TURNS={MAX_TURNS}")

In [ ]:
# Cell 6 — Stage 1: Speechify (clean → spoken VI)
!python -m src.speechify_run -d $DATASET --split $SPLIT --save_dir results_vi --llm_model_name $MODEL --max_train_samples $MAX_TRAIN_SAMPLES

In [ ]:
# Cell 7 — Stage 2+4: Synthesis (slot detection + turn-taking)
!python -m src.synthesis.run -d $DATASET -s $SPLIT --input_root results_vi --save_root outputs/vi_tt --llm_model_name $MODEL --boundary_model_name $MODEL --tt_model_name $MODEL --max_dialogues $MAX_DIALOGUES --max_turns $MAX_TURNS

In [ ]:
# Cell 8 — Stage 4 (cont.): Backchannel text
!python -m src.synthesis.run_add_bc --dataset $DATASET --split $SPLIT --input_root outputs/vi_tt --output_root outputs/vi_tt_bc --model_name $MODEL

In [ ]:
# Cell 9 — Verify Phase 1 outputs
!echo "=== results_vi ===" && find results_vi -type f | head -20
!echo "=== outputs/vi_tt ===" && find outputs/vi_tt -type f | head -20
!echo "=== outputs/vi_tt_bc ===" && find outputs/vi_tt_bc -type f | head -20
!ls -R outputs/vi_tt_bc 2>/dev/null | head -80

### ⏸️ STOP — Before Phase 2
1. **Save outputs**: Kaggle → `Save Version` (to keep `outputs/vi_tt_bc` as Dataset) or they stay in `/kaggle/working` if you just Restart.
2. **Kernel → Restart & Clear Output** (required — clears transformers 4.53 so Stage 5 can install its deps).
3. After restart, run Phase 2 cells below. If you saved as Dataset, mount it in `Settings → Add Input` on the new version.


---
## PHASE 2 — Stage 5 (OmniVoice TTS)
Run **after kernel restart**. Re-clone not needed if `/kaggle/working/Vilex` persisted; otherwise clone again.


In [ ]:
# Cell 11 — Re-enter repo + verify Phase 1 outputs still exist
import os
from pathlib import Path
if Path("/kaggle/working/Vilex").exists():
    os.chdir("/kaggle/working/Vilex")
elif Path("Vilex").exists():
    os.chdir("Vilex")
elif Path("/kaggle/input").exists():
    # If you saved Phase 1 as dataset, clone fresh
    import subprocess, sys
    if not Path("Vilex").exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--help"])
        print("Cloning fresh...")
        # !git clone https://github.com/thaiphu05/Vilex.git
        pass
print(f"CWD={os.getcwd()}")
!pwd && ls -lh
!ls outputs/vi_tt_bc/text_dialogue_interviewer/train/*.json 2>&1 | head -20
!find outputs -type f 2>/dev/null | head -20

In [ ]:
# Cell 11b — If outputs missing, restore from Kaggle Input dataset
# If you saved Phase 1 as dataset named 'vilex-phase1', uncomment:
# !cp -r /kaggle/input/vilex-phase1/outputs ./outputs 2>/dev/null || cp -r /kaggle/input/vilex-phase1/Vilex/outputs ./outputs
# !cp -r /kaggle/input/vilex-phase1/results_vi ./results_vi 2>/dev/null || true
# !ls -R outputs | head -40

In [ ]:
# Cell 12 — Install Stage 5 deps (OmniVoice)
# Note: torch already in Kaggle image. Install Stage5 shared + OmniVoice Block C.
!pip install -q -r requirements-stage5.txt
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q pyloudnorm
!python -c "import torch; print(f'torch={torch.__version__} cuda={torch.cuda.is_available()}')"
!python -c "import whisperx, silero_vad; print('whisperx+silero ok')"

In [ ]:
# Cell 13 — Voice pool path (EDIT THIS)
# Each *.wav must have sidecar *.txt with exact transcript.
# Examples:
#   Kaggle Dataset mount:  /kaggle/input/voice-clone
#   Working dir upload:    voice_clone  (upload via Kaggle → Add Input → Upload)
VOICE_POOL = "/kaggle/input/voice-clone"  # ← đổi thành đường dẫn thực tế của bạn
import os, glob
from pathlib import Path
if not os.path.isdir(VOICE_POOL):
    # try fallbacks
    for cand in ["voice_clone", "/kaggle/input/voice_clone", "/kaggle/working/voice_clone", "../voice_clone"]:
        if os.path.isdir(cand):
            VOICE_POOL = cand
            break
print(f"VOICE_POOL={VOICE_POOL}")
!ls -lh "$VOICE_POOL" 2>&1 | head -40
# Validate sidecars
import glob as _glob
wavs = sorted(_glob.glob(os.path.join(VOICE_POOL, "*.wav")))
print(f"Found {len(wavs)} wavs")
for w in wavs[:10]:
    txt = os.path.splitext(w)[0] + ".txt"
    ok = "✅" if os.path.isfile(txt) else "❌ MISSING .txt"
    print(f"{ok} {w} -> {txt}")
if len(wavs) < 2:
    print("ERROR: need >=2 wavs with .txt sidecars (convert_spoken.py:732)")

In [ ]:
# Cell 14 — Stage 5: OmniVoice render (VI default)
# --device cuda on Kaggle GPU, fallback to cpu if OOM
VOICE_POOL = "/kaggle/input/voice-clone"  # keep in sync with Cell 13
# Auto-fallback if path differs
import os
for cand in [VOICE_POOL, "voice_clone", "/kaggle/input/voice_clone"]:
    if os.path.isdir(cand):
        VOICE_POOL = cand
        break
print(f"Using VOICE_POOL={VOICE_POOL}")
!python tts_render/convert_spoken.py \
  --tts_backend omnivoice \
  --language vi \
  --input_glob "outputs/vi_tt_bc/text_dialogue_interviewer/train/*.json" \
  --save_dir outputs/vi_audio \
  --omnivoice_voice_pool "$VOICE_POOL" \
  --num_variants 1 \
  --device cuda \
  --max_dialogues 1

In [ ]:
# Cell 14b — If CUDA OOM, retry on CPU
# !python tts_render/convert_spoken.py --tts_backend omnivoice --language vi --input_glob "outputs/vi_tt_bc/text_dialogue_interviewer/train/*.json" --save_dir outputs/vi_audio --omnivoice_voice_pool "$VOICE_POOL" --num_variants 1 --device cpu --max_dialogues 1

In [ ]:
# Cell 15 — Preview + export
!find outputs/vi_audio -type f 2>/dev/null | head -30
!ls -lh outputs/vi_audio/text_dialogue_interviewer/train/*/var00/dialogues/dialogue.wav 2>/dev/null | head -5
try:
    from IPython.display import Audio, display
    import glob as _g
    wavs = _g.glob("outputs/vi_audio/**/dialogue.wav", recursive=True)
    if wavs:
        print(f"Preview: {wavs[0]}")
        display(Audio(wavs[0]))
    else:
        print("No dialogue.wav yet — check logs above")
except Exception as e:
    print(e)
!zip -r /kaggle/working/vi_audio.zip outputs/vi_audio 2>&1 | tail -20
!ls -lh /kaggle/working/vi_audio.zip 2>/dev/null

## Notes
- **Full run:** set `MAX_TRAIN_SAMPLES=""` / `MAX_DIALOGUES=""` (empty = all) in Cell 5 and remove `--max_dialogues 1` in Cell 14. Warning: LLM calls are rate-limited (`src/gemini_client.py:122` 13 s interval, ~5 req/min) — full dataset takes hours.
- **One-command alternative:** `./run_vi_pipeline.sh` does same 4 steps, but notebook cells give clearer logs per stage.
- **Output layout:** `outputs/vi_audio/text_dialogue_interviewer/train/<id>/var00/dialogues/dialogue.wav` (stereo, ch0=assistant ch1=user) + `meta.json` — see `docs/stage5-tts.md:38`.
- **Persist:** `/kaggle/working/vi_audio.zip` survives `Save Version → Output`. Download from Kaggle Output tab.
